# Week 6 Homework: Your Answers vs. Your Vibe-Coded Dashboard

**Week 4 dashboard link:** _(paste your Streamlit URL)_

---

Two weeks ago you vibe-coded a MovieLens dashboard before you had the tools to judge it. Since then you've had the stats lecture (Week 5) and this week's EDA lecture. Now you do it the real way: answer the same 4 questions yourself, by hand, then audit the dashboard against your own work.

**Ground rules**

- 🚫 **No AI for code or answers in Parts 1 and 2.** Lecture notebooks, pandas/seaborn docs, Google, and Stack Overflow are all fair game. Write Part 3 in your own words too.
- 🔒 **Lock in before you peek.** Finish Parts 1 and 2 (answers written down) *before* you open your dashboard. If you look first, you'll anchor on its numbers and the comparison is worthless.
- 🔢 **Show your numbers.** Every answer needs actual values (counts, averages, titles), not just a vibe.
- ✏️ Written answers go in markdown cells (double-click to edit, Shift+Enter to render). Most are 2 to 4 sentences. Short and specific beats long and vague.

## The data

`movie_ratings.csv` has one row per rating: 100,000 ratings from 943 users on 1,682 movies.

- `user_id`, `movie_id`: who rated, which movie
- `rating`: 1 to 5 stars
- `timestamp`, `rating_year`: when the rating was given
- `age`, `gender`, `occupation`, `zip_code`: the rater
- `title`: movie title, with release year in parentheses
- `year`, `decade`: the movie's release year and decade
- `genres`: all of the movie's genres in one string, separated by the pipe character (a movie can have several)

## Setup

In [61]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Works whether this notebook sits in the week folder or inside exercise/
data_path = next(p for p in ["data/movie_ratings.csv", "../data/movie_ratings.csv"] if Path(p).exists())
df = pd.read_csv(data_path)
df.head()

,user_id,movie_id,rating,timestamp,age,gender,occupation,zip_code,title,year,decade,genres,rating_year
0,196,242,3,1997-12-04 15:55:49,49,M,writer,55105,Kolya (1996),1996.0,1990.0,Comedy,1997
1,186,302,3,1998-04-04 19:22:22,39,F,executive,00000,L.A. Confidential (1997),1997.0,1990.0,Crime|Film-Noir|Mystery|Thriller,1998
2,22,377,1,1997-11-07 07:18:36,25,M,writer,40206,Heavyweights (1994),1994.0,1990.0,Children|Comedy,1997
3,244,51,2,1997-11-27 05:02:03,28,M,technician,80525,Legends of the Fall (1994),1994.0,1990.0,Drama|Romance|War|Western,1997
4,166,346,1,1998-02-02 05:33:16,47,M,educator,55113,Jackie Brown (1997),1997.0,1990.0,Crime|Drama,1998


---
# Part 1: Methodology

## 1.1 Know your data first

Before answering anything, check what you're working with: shape, column types, missing values, duplicate rows, and the range of the numeric columns.

In [68]:
# Your data checks here

print(df.shape) #shape



(100000, 13)


In [69]:
print(df.dtypes) #column and types

user_id          int64
movie_id         int64
rating           int64
timestamp          str
age              int64
gender             str
occupation         str
zip_code           str
title              str
year           float64
decade         float64
genres             str
rating_year      int64
dtype: object


In [70]:
print(df.isna().sum()) # what collumns have na

user_id         0
movie_id        0
rating          0
timestamp       0
age             0
gender          0
occupation      0
zip_code        0
title           0
year           30
decade         30
genres          0
rating_year     0
dtype: int64


In [71]:
print(df.duplicated()) #duplicated rows

0        False
1        False
2        False
3        False
4        False
         ...  
99995    False
99996    False
99997    False
99998    False
99999    False
Length: 100000, dtype: bool


In [72]:

print(df.min(numeric_only=True)," \n\n",df.max(numeric_only=True)) #min and max for coluimns

user_id           1.0
movie_id          1.0
rating            1.0
age               7.0
year           1922.0
decade         1920.0
rating_year    1997.0
dtype: float64  

 user_id         943.0
movie_id       1682.0
rating            5.0
age              73.0
year           1998.0
decade         1990.0
rating_year    1998.0
dtype: float64


**✏️ Data notes:** _(What did you find? Anything missing, duplicated, or surprising? What did you do about it, and why?)_

## 1.2 Decide before you chart

In Week 4 the questions were ambiguous on purpose. This is where *you* commit to an interpretation. Fill it in now, and update it if you change your mind while working (just say so).

**Q1 Genre breakdown**
- What you have to decide: What am I counting, ratings or movies? How do I handle movies that have several genres?
- My choice: include them into the bar chart
- Why: because a movie is defined by several genre and there exists graph to take into account several variables

**Q2 Genre satisfaction**
- What you have to decide: "Average rating" of what? How do I treat genres that have very little data?
- My choice: created a bar chart with a line indicating the overall mean , the genres with low rating are represented in one line
- Why: it saves space and shows the viewer successesfull genres 

**Q3 Ratings over time**
- What you have to decide: Is every year backed by the same amount of data? What do I do with movies that have no release year?
- My choice: Not every year is backed by the same amount of data. I excluded the movies that have no release year from this visualization.
- Why: the number of ratings changes yearly, some averages are more consistant than others, the movies without a release year are omited

**Q4 Best movies**
- What you have to decide: What is a movie's rating (the mean of what)? Is "at least 50" ≥ or >? What if there's a tie?
- My choice: A movie's rating is the mean of all user ratings, with a minimum of ≥ 50 ratings, and all movies tied for the highest mean rating are included.

- Why: Averaging user ratings and requiring at least 50 ratings reduces the influence of movies with limited data and makes the rankings more reliable.

---
# Part 2: Results (by hand, no AI)

Answer each question with a chart or table **and** a written answer with real numbers. Pick the chart type yourself and be ready to defend it in Part 3. Add as many code cells as you need.

## Q1. Genre breakdown

**What's the distribution of genres among the movies that were rated?**

<details>
<summary>Stuck on movies with several genres? Hint</summary>

`genres` is one string like `Crime|Film-Noir|Mystery|Thriller`. To get one row per genre:

```python
genre_df = df.assign(genre=df['genres'].str.split('|')).explode('genre')
```

Heads up: after this, a movie with 3 genres takes up 3 rows. Think about what that does to anything you count or average. Exploding is fine for "how often does a genre show up", but be careful calling the result "market share" (the percentages add up to more than 100%).
</details>

In [88]:
# Q1: your code here
df.head()
#df.groupby("genres")["rating"].mean().sort_values(ascending=False) original thought
print(df["genres"].str.split("|").explode().value_counts())


genres
Drama          39895
Comedy         29832
Action         25589
Thriller       21872
Romance        19461
Adventure      13753
Sci-Fi         12730
War             9398
Crime           8055
Children        7182
Horror          5317
Mystery         5245
Musical         4954
Animation       3605
Western         1854
Film-Noir       1733
Fantasy         1352
Documentary      758
unknown           10
Name: count, dtype: int64


**✏️ My answer (with numbers):** _(Which genres dominate, by how much, and what exactly did you count?)_

## Q2. Genre satisfaction

**Which genres have the highest average rating? Which have the lowest?**

Show how much data is behind each average (for example, the count next to it). An average of 10 ratings and an average of 10,000 ratings are not the same kind of evidence.

In [168]:
# Q2: your code here
seperated_genres_df = df.assign(genres=df["genres"].str.split("|").explode("genres"))
seperated_genres_df.groupby("genres")["rating"].mean().sort_values(ascending=False)

genres
Fantasy        3.585557
Crime          3.568008
Western        3.560268
Animation      3.552948
Horror         3.544402
War            3.541503
Musical        3.540009
Comedy         3.533258
Children       3.532895
Documentary    3.531486
Drama          3.531361
Sci-Fi         3.529293
Thriller       3.527932
Romance        3.523141
Adventure      3.521794
Action         3.510983
Film-Noir      3.504914
Mystery        3.504049
unknown        3.285714
Name: rating, dtype: float64

**✏️ My answer (with numbers):** _(Top and bottom genres, their averages, and how much data each one rests on.)_

## Q3. Ratings over time

**How has the mean rating changed across movie release years?**

Show how many ratings each year is based on, alongside the trend.

In [ ]:
# Q3: your code here

#rotdf  = seperated_genres_df.groupby(['genres', 'year'])['rating'].mean().unstack('genres') this would make a df 
seperated_genres_df.groupby(['genres', 'year'])["rating"].mean()

genres   year  
Action   1922.0    3.571429
         1931.0    4.500000
         1932.0    4.000000
         1933.0    4.200000
         1934.0    4.190476
                     ...   
unknown  1991.0    5.000000
         1993.0    3.000000
         1994.0    1.000000
         1995.0    3.000000
         1997.0    3.500000
Name: rating, Length: 1181, dtype: float64

**✏️ My answer (with numbers):** _(What's the trend? Is it equally believable across the whole timeline?)_

## Q4. Best movies, with a floor

**What are the 5 best-rated movies that have at least 50 ratings? What changes if you raise the floor to 150?**

In [ ]:
# Q4: your code here
#seperated_genres_df.groupby("genres")["rating"].mean()
#seperated_genres_df[seperated_genres_df["rating"]<=50]
filtered_by_rating_df = seperated_genres_df[seperated_genres_df.groupby('movie_id')['rating'].transform('count') >= 50]
#notes: groups all rows that share movie id while keeping their ratings and then counts and filters them based on the count
q450 = filtered_by_rating_df.groupby(["movie_id", "title"])["rating"].mean().sort_values(ascending=False).head(5)
#groups movies by their title and id's, then means the ratings and sorts 
print(q450,"\n")

filtered_by_rating_df = seperated_genres_df[seperated_genres_df.groupby('movie_id')['rating'].transform('count') >= 150]
q4150 = filtered_by_rating_df.groupby(["movie_id", "title"])["rating"].mean().sort_values(ascending=False).head(5)
print(q4150)

movie_id  title                                                 
408       Close Shave, A (1995)                                     4.491071
318       Schindler's List (1993)                                   4.466443
169       Wrong Trousers, The (1993)                                4.466102
483       Casablanca (1942)                                         4.456790
114       Wallace & Gromit: The Best of Aardman Animation (1996)    4.447761
Name: rating, dtype: float64 

movie_id  title                           
318       Schindler's List (1993)             4.466443
483       Casablanca (1942)                   4.456790
64        Shawshank Redemption, The (1994)    4.445230
603       Rear Window (1954)                  4.387560
12        Usual Suspects, The (1995)          4.385768
Name: rating, dtype: float64


**✏️ My answer (with numbers):** _(Both top-5 lists. Which movies dropped out or moved in when the floor went up, and what does that say about small samples?)_

> 🔒 **Stop here.** Parts 1 and 2 should be completely filled in before you open your dashboard.

---
# Part 3: Analysis

Now open your Week 4 dashboard.

## 3.1 How the dashboard got built

Paste 2 prompts that shaped it (from your build log, or from your AI chat history if you didn't keep a log; don't invent any). For each one: what did you ask, what did the AI give you first, and what did you change or accept without checking?

**Prompt 1**
- What I asked: I dont remember nore can i find it. i believe i asked it to include a bargraph
- AI's first attempt:it made a bar graph 
- What I changed (or accepted without checking):

**Prompt 2**
- What I asked: I also probably asked for a bar graph
- AI's first attempt:
- What I changed (or accepted without checking):

## 3.2 Compare the answers

Read the answer off your dashboard for each question and put it next to yours.

**Q1**
- Dashboard says: drama
- I say: drama
- Same? (yes / partly / no): yes
- If different, why?

**Q2**
- Dashboard says: film noir
- I say: action
- Same? (yes / partly / no): no
- If different, why? The AI probably did the average rating and number of ratings per genre

**Q3**
- Dashboard says: It created 2 types of graphs overlapping, the linegraph(mean) shows the change of movies over time the bar graph shows # of ratings over time
- I say: I showed the change in every genres over time
- Same? (yes / partly / no):no
- If different, why? The ai aimed to show a relationship between the mean and count while i tried to show the relation ship between the genres, years and mean ratings

**Q4**
- Dashboard says: close shave and shindlers list
- I say: close shave and shindlers list
- Same? (yes / partly / no):yes
- If different, why?

**When numbers differ, check these first.** Most mismatches come from one of these:

- Counting **ratings** vs. counting **movies**
- One row per rating vs. **one row per rating per genre** (after exploding)
- **Mean of all ratings** vs. the mean of each movie's average
- "At least 50" as ≥ vs. >, or applying the floor before vs. after another step
- What happened to movies with a **missing year**
- Genres with **tiny sample sizes** left in or filtered out

**✏️ Pick two questions and explain what's going on** (2 to 3 sentences each). Track down the *actual cause*: read your dashboard's code, or re-run it with one change. "The AI did it differently" is not an explanation. If the answers match, say why you now trust them.

## 3.3 Chart choices

**Q1**
- Chart the dashboard used:
- Chart I used:
- Which communicates the answer better, and why?

**Q2**
- Chart the dashboard used:
- Chart I used:
- Which communicates the answer better, and why?

**Q3**
- Chart the dashboard used:
- Chart I used:
- Which communicates the answer better, and why?

**Q4**
- Chart the dashboard used:
- Chart I used:
- Which communicates the answer better, and why?

Use what you learned about chart choice and misleading charts (Week 5) and this week's lecture. Is it sorted? Are the labels readable? Are the axes honest? Does it show how much data is behind each number?

## 3.4 What can this data actually tell you?

**Answer at least 3 of these 5**, using your own results as evidence.

**1. Who is in this data?** Check the fewest and the most ratings any single user has given (`df.groupby('user_id').size()`), and look at `rating_year`. What kind of person ends up in a dataset like this, and who is missing? How might that tilt your Q2 answer? *(Week 5: sampling bias)* 

**2. How sure are you about the bottom genre?** Take the lowest-rated genre from your Q2. How many ratings *and* how many distinct movies is it built on? Estimate a rough 95% interval: mean ± 1.96 × std / √n. Would you bet it's truly the worst genre? *(Week 5: standard error and confidence intervals. Also ask yourself: are 5,000 ratings of the same handful of movies really 5,000 independent pieces of evidence? Remember the one-row-per-model lesson from last week's homework.)*

the sample size for unkown movies is really small so its hard to truly get any information about those kinds of movies

**3. What can Q3 really say about time?** How many different values does `rating_year` have? So can this data tell you whether people's taste changed over time? What does your Q3 trend actually describe instead? Why might an old movie in this dataset score higher than a recent one? *(Think: who rates a 1940s film in 1997, and which old movies are even in the data?)*

this cant really tell us anything about taste, for example animated movies back then might have been recieved differently since the technology for them wasnt as adequate. Maybe all action movies now were just done by better producers, ect


**4. Does "average" depend on how you average?** Recompute Q2 two ways: (a) average over all ratings, (b) average each movie first, then average the movies within each genre. Does the top genre change? "Which genre do *people* like most?" and "which genre has the best *movies*?" are different questions. *(Week 5: confounding, Simpson's paradox)*

Using all ratings, [genre] ranked highest; averaging each movie first, [genre] ranked highest, so the top genre [changed/stayed the same]. The first method gives more weight to movies with more ratings, while the second treats every movie equally.

**5. Deployment risk.** A studio exec uses your dashboard to decide which genre to green-light. What's the most misleading conclusion they could draw from one of your charts? Write the one-line caveat you'd put under that chart.

**✏️ Your answers:** _(label each one with its number)_

## 3.5 Verdict

**Where I'd override the AI** (something it got wrong, hid, or did poorly, and what your version did better):

**Where the AI held up** (something it did well, or a choice you doubted at first that turned out right):

**One sentence:** looking back at both weeks, who was driving, you or the AI, and when?

---
## Optional stretch

Only if you want more practice:

- Pick 4 genres. How does the mean rating change as viewer age increases? (Try more than 4 and see which genres make the most interesting chart.)
- Plot number of ratings vs. mean rating for each movie. Do the most-rated movies get higher ratings? Is there a correlation? *(Week 5: correlation vs. causation)*

---
## Before you submit

- [ ] Copy this notebook into the `exercise/` folder and rename it `Firstname_Lastname_week_06_exercise.ipynb`
- [ ] **Kernel → Restart & Run All** and make sure it runs top to bottom with your charts showing
- [ ] Every ✏️ cell and every fill-in line is filled in
- [ ] Open a PR from your fork, same as every other week